# Why adaptive weighting matters — RBA on a boundary-layer problem

**RBA (Residual-Based Attention)** gives every collocation point its own weight
$\lambda_i$, updated from the residual history:
$$\lambda_i \leftarrow \gamma\,\lambda_i + \eta\,\frac{|r_i|}{\max_j |r_j|},\qquad
L_{pde} = \frac{1}{N}\sum_i (\lambda_i\, r_i)^2$$
Points whose residual *stays* large accumulate weight (up to $\eta/(1-\gamma)$); easy points
fade. The optimizer is forced to work on the hard region. (This is the `rba: true` flag in
underPINN's loss config.)

**The toy problem** — a singularly perturbed convection–diffusion BVP:
$$\varepsilon\,u'' - u' = 0,\qquad u(0)=0,\ u(1)=1,\qquad \varepsilon = 0.02$$
$$u_{exact}(x) = \frac{e^{(x-1)/\varepsilon} - e^{-1/\varepsilon}}{1 - e^{-1/\varepsilon}}
\quad\text{— flat } u\approx 0 \text{, then a boundary layer of width } \varepsilon \text{ at } x=1.$$

**Why the plain PINN fails here:** $u\approx 0$ satisfies the ODE *exactly* on 98% of the
domain. The mean loss is tiny for this wrong 'outer' solution, and the optimizer settles
there — stuck at ~50% error indefinitely. RBA notices the stubborn residual near $x=1$,
piles weight on it, and resolves the layer.

Runs in ~1–2 minutes total on Colab (CPU or GPU).

In [ ]:
# Cell 1 -- Problem setup
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

EPS = 0.02   # layer width; try 0.05 (easier) or 0.01 (too hard for both!)

def u_exact(x):
    return (torch.exp((x-1)/EPS) - np.exp(-1/EPS)) / (1 - np.exp(-1/EPS))

xg = torch.linspace(0, 1, 2001, device=device).reshape(-1, 1)   # evaluation grid
x0 = torch.tensor([[0.0]], device=device); x1 = torch.tensor([[1.0]], device=device)

plt.figure(figsize=(8, 3))
plt.plot(xg.cpu().ravel(), u_exact(xg).cpu().ravel(), 'g', lw=2)
plt.title(f'Exact solution: flat almost everywhere, boundary layer of width {EPS} at x=1')
plt.xlabel('x'); plt.ylabel('u'); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## The math behind the layer — and the training deadlock

**Where the exact solution comes from.** The ODE is linear with constant coefficients, so try
$u = e^{mx}$. The characteristic equation is
$$\varepsilon m^2 - m = 0 \;\Rightarrow\; m = 0 \ \text{ and } \ m = 1/\varepsilon,$$
so the general solution is $u(x) = A + B\,e^{x/\varepsilon}$. Applying $u(0)=0,\ u(1)=1$ gives
the formula coded above. Note the two ingredients: a **constant** ($m=0$) and an
**enormously fast exponential** ($m = 1/\varepsilon = 50$).

**Why the layer has width $\varepsilon$.** The term $e^{(x-1)/\varepsilon}$ is controlled by
the distance to the right wall:

| position | value |
|---|---|
| $x = 1$ | $e^{0} = 1$ |
| $x = 1-\varepsilon$ | $e^{-1} \approx 0.37$ |
| $x = 1-5\varepsilon$ | $e^{-5} \approx 0.007$ — gone |

So the solution is indistinguishable from 0 until within a few $\varepsilon$ of $x=1$.

**The physical balance argument (singular perturbation).** Read the ODE as steady
convection–diffusion: $u'$ is convection sweeping the profile toward the wall, $\varepsilon u''$
is weak diffusion.
- **Outer region:** drop the tiny diffusion term → $u' = 0$ → $u = $ const, and $u(0)=0$
  forces $u \approx 0$. But dropping $\varepsilon u''$ lost a derivative, so this reduced
  solution *cannot* also satisfy $u(1)=1$.
- **Inner region:** the jump from 0 to 1 must happen over some width $\delta$, where
  $u' \sim 1/\delta$ and $u'' \sim 1/\delta^2$. The two terms balance when
  $$\frac{\varepsilon}{\delta^2} \sim \frac{1}{\delta} \;\Rightarrow\; \boxed{\delta \sim \varepsilon}.$$
  Diffusion only "turns on" in a region exactly thin enough to matter.

**Why this deadlocks a plain PINN.** The outer solution $u \equiv 0$ satisfies the ODE
*exactly* and matches $u(0)=0$; only $u(1)=1$ objects. Now count points: with $N=1000$
uniform collocation points and $\varepsilon = 0.02$, only ~20 sit inside the layer. For a
flat network solution the residual is ~0 at the other ~980 points, so the **mean** loss is
tiny — the PINN at 50% error *looks* converged. Worse, building the required gradient
$u' \sim 1/\varepsilon = 50$ temporarily *raises* residuals at neighbouring points, so the
mean loss punishes every attempt to fix things. The lone BC term pulls up, 980 easy points
push back: a stable local minimum.

**How RBA breaks it.** The update $\lambda_i \leftarrow \gamma\lambda_i + \eta|r_i|/\max|r|$
has two limits:
- easy points: $|r_i| \approx 0$ → $\lambda_i \leftarrow \gamma\lambda_i$ — geometric decay
  toward 0; the 980 outer points progressively lose their vote;
- stubborn points: $|r_i| \approx \max|r|$ every epoch → $\lambda_i \to \eta/(1-\gamma) = 10$.

Since the loss uses $(\lambda_i r_i)^2$, a layer point ends up counting up to
$10^2 = 100\times$ an average point — precisely cancelling the 980-vs-20 head-count
imbalance that created the deadlock. The optimizer can now afford the temporary residual
spike needed to carve out the steep gradient, and the run below drops from L2 ≈ 0.53 to
≈ 0.006 with the same budget.

In [ ]:
# Cell 2 -- Shared machinery: network, residual, and one training loop with an RBA switch
def make_mlp():
    torch.manual_seed(0)   # identical initial weights for a fair comparison
    return nn.Sequential(nn.Linear(1, 64), nn.Tanh(),
                         nn.Linear(64, 64), nn.Tanh(),
                         nn.Linear(64, 64), nn.Tanh(),
                         nn.Linear(64, 1)).to(device)

# FIXED collocation points — RBA tracks a weight per point, so points must persist
torch.manual_seed(1)
N = 1000
xc = torch.rand(N, 1, device=device)

def residual(model, x):
    x = x.detach().requires_grad_(True)
    u = model(x)
    u_x  = torch.autograd.grad(u,   x, torch.ones_like(u),   create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    return EPS*u_xx - u_x                     # eps*u'' - u' = 0

def train(use_rba, epochs=8000, gamma=0.999, eta=0.01, tag=''):
    model = make_mlp()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    lam = torch.ones(N, 1, device=device)     # per-point weights (all start equal)
    hist = {'epoch': [], 'l2': []}
    t0 = time.perf_counter()
    for e in range(epochs):
        opt.zero_grad()
        r = residual(model, xc)
        if use_rba:
            # --- the whole RBA algorithm: two lines ---
            with torch.no_grad():
                lam = gamma*lam + eta * r.abs() / (r.abs().max() + 1e-12)
            loss_pde = ((lam * r)**2).mean()
        else:
            loss_pde = (r**2).mean()
        loss_bc = (model(x0) - 0.0)**2 + (model(x1) - 1.0)**2
        (loss_pde + 100*loss_bc.sum()).backward(); opt.step()
        if e % 100 == 0 or e == epochs-1:
            with torch.no_grad():
                err = torch.sqrt(torch.mean((model(xg) - u_exact(xg))**2)).item()
            hist['epoch'].append(e); hist['l2'].append(err)
    if device.type == 'cuda': torch.cuda.synchronize()
    print(f'{tag}: {time.perf_counter()-t0:.1f} s  |  final L2 error = {hist["l2"][-1]:.4f}')
    return model, lam, hist

In [ ]:
# Cell 3 -- Same network init, same points, same budget. Only the weighting differs.
model_plain, _,   hist_plain = train(use_rba=False, tag='plain PINN (mean loss)')
model_rba,   lam, hist_rba   = train(use_rba=True,  tag='RBA PINN             ')

In [ ]:
# Cell 4 -- Results: solution, convergence, and where the weights went
xp = xg.cpu().ravel(); ue = u_exact(xg).cpu().ravel()
with torch.no_grad():
    up = model_plain(xg).cpu().ravel(); ur = model_rba(xg).cpu().ravel()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].plot(xp, ue, 'g', lw=2.2, label='exact')
ax[0].plot(xp, up, 'b--', lw=1.8, label=f'plain (L2={hist_plain["l2"][-1]:.2f})')
ax[0].plot(xp, ur, 'r-.', lw=1.8, label=f'RBA (L2={hist_rba["l2"][-1]:.4f})')
ax[0].set_title('Plain settles for the easy outer solution')
ax[0].set_xlabel('x'); ax[0].set_ylabel('u'); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].semilogy(hist_plain['epoch'], hist_plain['l2'], 'b', label='plain')
ax[1].semilogy(hist_rba['epoch'],   hist_rba['l2'],   'r', label='RBA')
ax[1].set_title('L2 error: plain is stuck, RBA escapes')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('L2 error'); ax[1].legend(); ax[1].grid(alpha=.3, which='both')

ax[2].scatter(xc.cpu().ravel(), lam.cpu().ravel(), s=6, c='crimson', alpha=.6)
ax[2].set_title('Final RBA weights: they FOUND the boundary layer')
ax[2].set_xlabel('collocation point x'); ax[2].set_ylabel('weight  $\\lambda_i$'); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Takeaways

- **The failure mode is silent.** The wrong 'outer' solution $u\approx 0$ has a *tiny mean
  residual* — the plain loss looks converged while the answer is 50% wrong. Loss values
  alone can lie; adaptive schemes attack exactly this.
- **RBA is two lines of code.** An exponential moving average of the normalised residual
  per point: $\lambda_i \leftarrow \gamma\lambda_i + \eta |r_i|/\max|r|$. Weights are bounded
  by $\eta/(1-\gamma)$ (=10 here), so training stays stable.
- **The weights are interpretable.** Plot $\lambda(x)$ and you get a free error indicator —
  it spikes exactly at the boundary layer, like adaptive mesh refinement without a mesh.
- **Same idea, bigger problems.** Sharp fronts (Burgers, shocks), stiff reaction terms
  (Allen–Cahn) and boundary layers all benefit; this is why underPINN exposes `rba: true`
  as a one-flag option in its loss config.

**Experiments to try:** `EPS = 0.05` (plain eventually succeeds — RBA just gets there ~4×
sooner) and `EPS = 0.01` (both fail — adaptivity is not magic; that regime needs
continuation/annealing in ε). Vary `gamma`/`eta` (the weight cap is $\eta/(1-\gamma)$).
Try resampling collocation points each epoch and watch RBA break — the weights need
persistent point identities.